# Safehouse Performance & Outcome Benchmarking — House of Hope

**Full ML pipeline:** Problem framing → Data prep → EDA → Modeling (Ridge benchmark) → ONNX export → Deployment in `SafehousePerformanceService` + Reports & Analytics UI.

**Data:** `ml-pipelines/lighthouse_csv_v7/` — `safehouses.csv`, `residents.csv`, `process_recordings.csv`, `home_visitations.csv`, `intervention_plans.csv`, `education_records.csv`, `health_wellbeing_records.csv`

**Run from:** repository `ml-pipelines/` directory so `lighthouse_csv_v7` resolves.

---

## Sections

1. Problem Framing  
2. Data Acquisition, Preparation & Exploration  
3. Outcome index & operational features  
4. Modeling & evaluation (Ridge + CV)  
5. Causal / explanatory interpretation  
6. ONNX export & deployment notes  
7. Limitations


## 1. Problem Framing

### Business problem

Leadership runs multiple safehouses with limited supervision capacity. They need to know **which locations are delivering favorable outcomes for girls** (reintegration progress, education, health, stability) and **whether lower performance might reflect caseload difficulty or weaker service intensity** (counseling documentation, home visits, intervention closure).

### Predictive vs explanatory

| Goal | Approach | Deliverable |
|------|----------|-------------|
| **Explanatory** | Ridge regression on standardized operational features | Signed coefficients: what practice patterns co-vary with outcomes |
| **Predictive / operational** | Same model produces **expected outcome** for each site’s observed practice intensity + caseload stress | **Gap = actual − expected** flags under/over-performance vs peers |

We do **not** claim that increasing process recordings *causes* better reintegration — the model is a **benchmark** for peer comparison, not a causal proof.

### Success metrics

- Train MAE / R² on the full safehouse panel (small N — see limitations)
- Cross-validated MAE (high variance with n=9)
- Deployment: ONNX parity check + API integration


## 2. Imports & reproducible pipeline

Core logic lives in `scripts/safehouse_train_export.py` (single source of truth for training + export). Re-run that script after data updates.


In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("lighthouse_csv_v7")
assert DATA_DIR.is_dir(), "Run from ml-pipelines/ or set DATA_DIR"

from scripts.safehouse_train_export import (
    load_tables,
    build_safehouse_frames,
    train_and_export,
    FEATURE_COLS,
)

sh, res, pr, hv, ip, edu, hlth = load_tables()
df = build_safehouse_frames(sh, res, pr, hv, ip, edu, hlth)
print(df[["safehouse_id", "name", "outcome_index"] + FEATURE_COLS].round(3).to_string())

### 2.1 Outcome index (0–100)

Composite label aligned with production C# (`SafehousePerformanceService.ComputeOutcomeIndex`):

- 30/85 × % reintegration status = Completed  
- 22/85 × mean education `progress_percent` (capped at 100)  
- 18/85 × normalized `general_health_score` (1–5 → 0–100)  
- 15/85 × % residents with current risk = Low  

`initial_risk_level` is **not** in the deployed EF schema; we do not use risk-delta in the label so training matches production scoring.


In [ ]:
import matplotlib.pyplot as plt

ax = df.plot.scatter(
    x="process_per_resident", y="outcome_index",
    figsize=(6, 4), title="Process intensity vs outcome index"
)
plt.ylabel("Outcome index")
plt.tight_layout()
plt.show()

ax2 = df.plot.scatter(
    x="pct_high_critical_risk", y="outcome_index",
    figsize=(6, 4), color="coral", title="High/Critical risk share vs outcome"
)
plt.ylabel("Outcome index")
plt.tight_layout()
plt.show()

## 3. Train Ridge + export ONNX

Pipeline: `SimpleImputer(median)` → `StandardScaler` → `Ridge(alpha=2)`.

Exports:
- `safehouse_performance_model.onnx` — full pipeline, input `float_input` `[1, 6]`
- `safehouse_performance_preprocessing.json` — tier cuts, network means for narrative, metrics

**Deploy:** copy both files to `backend/Models/` (see `SafehousePerformanceService.cs`).


In [ ]:
stats = train_and_export(df)
print("Train MAE:", round(stats["mae"], 4), " R2:", round(stats["r2"], 4))
print("Coefficients (scaled features):", {k: round(v, 4) for k, v in stats["coef_map"].items()})

## 4. Interpretation (explanatory)

- Positive coefficients on `process_per_resident` and `visits_per_resident` suggest that **documented counseling and family engagement intensity** co-vary with better composite outcomes *holding other features fixed* — consistent with supervision quality, **not** proof of causality.
- Negative coefficient on `caseload_complexity` and `pct_high_critical_risk` reflects **harder caseloads** — sites with more trauma flags or high/critical risk shares tend to have lower outcome index *all else equal*.
- **Gap analysis:** if actual outcome ≫ benchmark, the site may be outperforming expectations for its operational profile; if actual ≪ benchmark, leadership should review staffing, supervision, and whether documentation reflects real service delivery.

## 5. Deployment notes

- **API:** `GET /api/ML/safehouse-performance` (admin) — returns list of `SafehousePerformanceInsight`.
- **UI:** `frontend/src/pages/ReportsAnalytics.tsx` — cards + tier badges.
- **Privacy:** aggregate safehouse-level metrics only; no resident identifiers in API response names beyond internal analytics.

## 6. Limitations

- **Small n safehouses** — cross-validation MAE is unstable; coefficients are directional, not precise policy elasticities.
- **Outcome index** is a weighted composite — weights are judgmental; sensitivity analysis recommended before strategic funding shifts.
- **Incident data** per safehouse is not in the slim EF `IncidentReport` entity; monthly incident aggregates from CSV were not used in deployment features to avoid train/serve skew.
